# Task 4 - Proxy Target Variable Engineering

## Objective
In this task, we construct a proxy credit risk target variable using RFM (Recency, Frequency, Monetary) analysis and K-Means clustering. Since the dataset has no default label, we identify disengaged customers and label them as high-risk through RFM analysis and K-Means clustering.


In [1]:
# Import Libraries
import sys
import os

# Add project root
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load Dataset (Code Cell)
df = pd.read_csv("../data/raw/data.csv")

df.head()

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,ChannelId,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,ChannelId_3,1000.0,1000,2018-11-15T02:18:49Z,2,0
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-20.0,20,2018-11-15T02:19:08Z,2,0
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,ChannelId_3,500.0,500,2018-11-15T02:44:21Z,2,0
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,ChannelId_3,20000.0,21800,2018-11-15T03:32:55Z,2,0
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,ChannelId_2,-644.0,644,2018-11-15T03:34:21Z,2,0


In [ ]:
# Convert Transaction Time 
df["TransactionStartTime"] = pd.to_datetime(df["TransactionStartTime"])

In [ ]:
# Define Snapshot Date 
snapshot_date = df["TransactionStartTime"].max() + pd.Timedelta(days=1)
snapshot_date

Timestamp('2019-02-14 10:01:28+0000', tz='UTC')

In [ ]:
# Create RFM Table 
rfm = df.groupby("CustomerId").agg(
    Recency=(
        "TransactionStartTime",
        lambda x: (snapshot_date - x.max()).days
    ),
    Frequency=("TransactionId", "count"),
    Monetary=("Amount", "sum")
)

rfm.head()

,Recency,Frequency,Monetary
CustomerId,,,
CustomerId_1,84,1,-10000.0
CustomerId_10,84,1,-10000.0
CustomerId_1001,90,5,20000.0
CustomerId_1002,26,11,4225.0
CustomerId_1003,12,6,20000.0


In [6]:
# RFM Distribution Check 
rfm.describe()

,Recency,Frequency,Monetary
count,3742.000000,3742.000000,3.742000e+03
mean,31.461251,25.564404,1.717377e+05
std,27.118932,96.929602,2.717305e+06
min,1.000000,1.000000,-1.049000e+08
25%,6.000000,2.000000,4.077438e+03
50%,25.000000,7.000000,2.000000e+04
75%,54.000000,20.000000,7.996775e+04
max,91.000000,4091.000000,8.345124e+07


In [7]:
# Scale RFM Features
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm)

In [8]:
# KMeans Clustering 
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

rfm["cluster"] = kmeans.fit_predict(rfm_scaled)

rfm.head()

,Recency,Frequency,Monetary,cluster
CustomerId,,,,
CustomerId_1,84,1,-10000.0,0
CustomerId_10,84,1,-10000.0,0
CustomerId_1001,90,5,20000.0,0
CustomerId_1002,26,11,4225.0,1
CustomerId_1003,12,6,20000.0,1


In [9]:
# Cluster Analysis (Code Cell)
cluster_summary = rfm.groupby("cluster").mean()
cluster_summary

,Recency,Frequency,Monetary
cluster,,,
0,61.877279,7.720196,8.172068e+04
1,12.726566,34.800000,2.725741e+05
2,29.000000,4091.000000,-1.049000e+08


In [11]:
# Identify High-Risk Cluster 
high_risk_cluster = cluster_summary.sort_values(
    by=["Frequency", "Monetary"],
    ascending=True
).index[0]

high_risk_cluster

np.int32(0)

In [12]:
# Assign Target Variable 
rfm["is_high_risk"] = (rfm["cluster"] == high_risk_cluster).astype(int)

rfm["is_high_risk"].value_counts()

is_high_risk
0    2316
1    1426
Name: count, dtype: int64

In [13]:
# Merge with Original Data 
customer_df = df.groupby("CustomerId").agg(
    total_transaction_amount=("Amount", "sum"),
    avg_transaction_amount=("Amount", "mean"),
    transaction_count=("TransactionId", "count")
).reset_index()

customer_df = customer_df.merge(
    rfm[["is_high_risk"]],
    left_on="CustomerId",
    right_index=True,
    how="left"
)

customer_df.head()

,CustomerId,total_transaction_amount,avg_transaction_amount,transaction_count,is_high_risk
0,CustomerId_1,-10000.0,-10000.000000,1,1
1,CustomerId_10,-10000.0,-10000.000000,1,1
2,CustomerId_1001,20000.0,4000.000000,5,1
3,CustomerId_1002,4225.0,384.090909,11,0
4,CustomerId_1003,20000.0,3333.333333,6,0


In [14]:
# Final Validation 
customer_df["is_high_risk"].value_counts()

is_high_risk
0    2316
1    1426
Name: count, dtype: int64

In [15]:
# Save Output Dataset 
customer_df.to_csv(
    "../data/processed/customer_with_target.csv",
    index=False
)
print("/data/processed/customer_with_target.csv saved successfully.")

/data/processed/customer_with_target.csv saved successfully.



### Work Completed

* Calculated Recency, Frequency, and Monetary (RFM) metrics for each customer.
* Standardized RFM features using StandardScaler.
* Applied K-Means clustering with 3 clusters and a fixed random state for reproducibility.
* Analyzed cluster profiles to identify the least engaged customer segment.
* Created a binary target variable (`is_high_risk`) where:

  * 1 = High-risk customer
  * 0 = Low-risk customer
* Merged the target variable into the customer-level feature dataset.
* Validated target distribution and prepared the dataset for supervised model training.

### Outcome

A processed customer-level dataset was produced containing engineered features and the proxy credit risk target variable required for model development.

# Final Insight
## Key Insight

-  Using RFM-based clustering, customers were segmented into 3 behavioral groups. 
-  The least engaged cluster, characterized by low transaction frequency and low monetary value, was labeled as high-risk (is_high_risk = 1).
-  This proxy target will be used for supervised credit risk modeling in subsequent tasks.